# Feature Set v3 — wide, multi-station

Builds `mlo.features.weather_daily_v3` from bronze. Runs after `10_ingest_nightly`.

**Target is unchanged: `Bad` at t+1** — whether Midway records `PRCP > 0.5mm` *tomorrow*, from
`mlo.features.weather_labels`. Everything measured on day D is therefore a legitimate feature,
including Midway's own precipitation that day. The only hard rule is that nothing from D+1 onward
enters a feature column.

(Note: `train_baseline.py` and `MODEL_CARD.md` define `Bad` *same-day*, with no shift. Those are the
pre-Databricks artifacts and their metrics are not comparable to anything here. Benchmark v3 against
the MLflow runs `03_baseline_model.ipynb` logs, which are t+1 for both v1 and v2.)

## What makes v3 different

v1 and v2 are **long**: one row per station-day, Midway only. v3 is **wide**: one row per *date*,
with other stations pivoted into prefixed columns. That shape is what makes upstream signal usable —
Midwest systems track roughly west to east, so conditions at Rockford or Dubuque today carry
information about Midway tomorrow, which a long table can't express without a self-join.

**`station` changes meaning.** In v1/v2 it identifies who measured the row. In v3 every row is a
prediction *for* Midway, so `station` is a constant. It's retained only so v3 keeps the same
primary-key shape as v1/v2 and `03`'s `FeatureLookup` needs no structural change. Provenance lives
in the column prefixes (`RFD_PRCP_lag1`) instead.

## 0. Parameters

In [0]:
dbutils.widgets.text('catalog', 'mlo', 'Unity Catalog catalog')
dbutils.widgets.text('bronze_schema', 'weather_mlops', 'Bronze schema')
dbutils.widgets.text('features_schema', 'features', 'Feature schema')

In [0]:
import math

from databricks.feature_engineering import FeatureEngineeringClient
from pyspark.sql import Window
from pyspark.sql import functions as F

fe = FeatureEngineeringClient()

CATALOG = dbutils.widgets.get('catalog')
BRONZE = f"{CATALOG}.{dbutils.widgets.get('bronze_schema')}.noaa_historical_daily"
FT3 = f"{CATALOG}.{dbutils.widgets.get('features_schema')}.weather_daily_v3"

PRIMARY_STATION = 'GHCND:USW00014819'  # Midway — the prediction target

# IATA codes, so a column name says where the reading came from.
STATION_CODES = {
    'GHCND:USW00014819': 'MDW',  # Chicago Midway            -- target
    'GHCND:USW00094846': 'ORD',  # Chicago O'Hare            -- 26 km
    'GHCND:USW00004838': 'PWK',  # Chicago Executive         -- 37 km
    'GHCND:USW00094822': 'RFD',  # Rockford                  -- 125 km WNW
    'GHCND:USW00014839': 'MKE',  # Milwaukee Mitchell        -- 125 km N
    'GHCND:USW00094908': 'DBQ',  # Dubuque                   -- 260 km W
    'GHCND:USW00014922': 'MSP',  # Minneapolis-St Paul       -- 570 km NW
    'GHCND:USW00093819': 'IND',  # Indianapolis              -- 252 km SSE
    'GHCND:USW00094815': 'AZO',  # Kalamazoo                 -- 179 km E
    'GHCND:USW00014840': 'MKG',  # Muskegon                  -- 187 km ENE
    'GHCND:USW00014834': 'JOT',  # Joliet                    -- excluded, see below
}

FULL = ['AWND', 'PRCP', 'TMAX', 'TMIN', 'WDF2', 'WDF5', 'WSF2', 'WSF5', 'SNWD',
        'WT01', 'WT02', 'WT03']

# Which columns each station contributes. Kept as an explicit dict so the feature scope is
# reviewable in a diff rather than buried in the pivot logic below.
#
# The budget that drives this: ~1,827 usable dates with ~26% positive, so roughly 475 minority
# events. Logistic regression wants 10-20 events per feature, putting the ceiling near 30-60
# features INCLUDING engineered ones. All 11 stations x 12 datatypes would be 132 before a single
# lag. So the target station gets everything and the rest are scoped to what they plausibly add.
FEATURE_SPEC = {
    PRIMARY_STATION:      FULL,                       # MDW -- the target, gets everything
    'GHCND:USW00094822': ['PRCP', 'TMAX', 'AWND'],    # RFD -- upstream WNW, full instrument suite
    'GHCND:USW00014922': ['PRCP', 'TMAX', 'AWND'],    # MSP -- upstream NW, clipper track
    'GHCND:USW00014839': ['PRCP', 'TMAX', 'AWND'],    # MKE -- N, lake corridor
    'GHCND:USW00094908': ['PRCP', 'TMAX'],            # DBQ -- upstream W; wind EXCLUDED, see below
    'GHCND:USW00094846': ['PRCP'],                    # ORD -- near-duplicate of MDW, one column only
    'GHCND:USW00093819': ['PRCP'],                    # IND -- downstream SSE, weak but cheap
}

# Deliberately absent, and why:
#
#   PWK  37 km from MDW. Collinear with MDW/ORD; adds columns, not information.
#   AZO  downstream E. Receives Chicago's weather rather than leading it.
#   MKG  downstream ENE, same reasoning.
#   JOT  Joliet. Only 578 rows (starts 2024-11-16), no wind, no flags, and degrading further.
#        Retained in bronze as a data-quality case; useless as a feature.
#
#   DBQ wind (AWND/WDF2/WDF5/WSF2/WSF5). 13% null across the full record, but ~100% null in
#        recent months -- the loss is concentrated at the recent end, so those columns exist in
#        training and vanish at inference. Textbook training/serving skew. Re-include only if the
#        monthly null check shows it recovered.

UPSTREAM = ['RFD', 'MSP', 'MKE', 'DBQ']  # lagged 1-2 days for west-to-east lead time

print(f'{len(FEATURE_SPEC)} stations feeding v3')
print('raw feature columns:', sum(len(c) for c in FEATURE_SPEC.values()))

## 1. Pivot bronze to wide

One row per date. Each station contributes its scoped columns under an IATA prefix.

The join is `outer` on purpose: a date where one station reported nothing should still produce a row
with nulls, rather than silently vanishing from the feature table. Section 4 measures what that
costs.

In [0]:
bronze = spark.table(BRONZE)

frames = []
for station_id, cols in FEATURE_SPEC.items():
    code = STATION_CODES[station_id]
    sdf = bronze.filter(F.col('station') == station_id).select(['date'] + cols)
    for c in cols:
        sdf = sdf.withColumnRenamed(c, f'{code}_{c}')
    frames.append(sdf)

wide = frames[0]
for frame in frames[1:]:
    wide = wide.join(frame, on='date', how='outer')

print(f'{wide.count()} dates, {len(wide.columns)} columns after pivot')

## 2. Engineered features

Four transformations, each for a specific reason:

**Wind direction → `sin`/`cos`.** `WDF2`/`WDF5` are compass degrees. 359° and 1° are two degrees
apart in reality and 358 apart numerically, so feeding raw degrees to a linear model is actively
wrong. Each becomes a sin/cos pair and the raw column is dropped — it's still in bronze if needed.

**Weather flags → `fillna(0)`.** NOAA writes `WT01 = 1` on days fog occurred and *nothing* otherwise.
Null means "did not happen", so 0 is the correct fill. Dropping nulls here would delete ~86% of rows
on `WT03`.

**`SNWD` → `fillna(0)`.** Midway reports snow depth only when there is snow, so it is null ~65% of
the record by construction. Same reasoning as the flags.

**Lags and rolling windows.** Midway's own recent history, plus 1–2 day lags on upstream
precipitation — that lag is the entire point of the extra stations, since it takes systems roughly a
day to cover the distance.

**Seasonality.** Day-of-year as a sin/cos pair. Two columns for a strong signal in a dataset where
precipitation is plainly seasonal.

In [0]:
# One global time series now (the station dimension became columns), so the window has no
# partition. Spark will warn about a single partition; at ~2.4k rows that is fine and correct.
w = Window.orderBy('date')

# --- wind direction: degrees are circular, encode as sin/cos ---------------------------
for c in ('WDF2', 'WDF5'):
    rad = F.radians(F.col(f'MDW_{c}'))
    wide = (wide
            .withColumn(f'MDW_{c}_sin', F.sin(rad))
            .withColumn(f'MDW_{c}_cos', F.cos(rad))
            .drop(f'MDW_{c}'))

# --- event flags and snow depth: null means "did not happen" ---------------------------
zero_fill = [c for c in wide.columns if '_WT' in c] + ['MDW_SNWD']
wide = wide.fillna(0, subset=zero_fill)

# --- Midway's own recent history ------------------------------------------------------
for c in ('PRCP', 'TMAX', 'TMIN', 'AWND'):
    wide = wide.withColumn(f'MDW_{c}_lag1', F.lag(f'MDW_{c}', 1).over(w))

wide = (wide
        .withColumn('MDW_PRCP_roll3', F.avg('MDW_PRCP').over(w.rowsBetween(-2, 0)))
        .withColumn('MDW_PRCP_roll7', F.avg('MDW_PRCP').over(w.rowsBetween(-6, 0)))
        .withColumn('MDW_TMAX_roll3', F.avg('MDW_TMAX').over(w.rowsBetween(-2, 0))))

# --- upstream lead time: conditions to the west, 1-2 days back ------------------------
for code in UPSTREAM:
    if f'{code}_PRCP' not in wide.columns:
        continue
    for n in (1, 2):
        wide = wide.withColumn(f'{code}_PRCP_lag{n}', F.lag(f'{code}_PRCP', n).over(w))

# --- seasonality ----------------------------------------------------------------------
doy = F.dayofyear('date')
wide = (wide
        .withColumn('doy_sin', F.sin(2 * F.lit(math.pi) * doy / 365.25))
        .withColumn('doy_cos', F.cos(2 * F.lit(math.pi) * doy / 365.25)))

# --- primary key ----------------------------------------------------------------------
# Constant by design: every row is a prediction FOR Midway, not a measurement BY Midway.
feature_cols = [c for c in wide.columns if c != 'date']
v3 = (wide
      .withColumn('station', F.lit(PRIMARY_STATION))
      .select('station', 'date', *feature_cols))

print(f'{len(feature_cols)} feature columns')
for c in sorted(feature_cols):
    print('   ', c)

## 3. Write v3

No `DROP TABLE`. `01` and `02` drop before writing, which resets the Delta transaction log to
version 0 every run and throws away the history — the thing that makes "we version our data"
mean something. Here the table is created once and overwritten thereafter, so every nightly
rebuild is a new readable version:

```sql
DESCRIBE HISTORY mlo.features.weather_daily_v3;
SELECT * FROM mlo.features.weather_daily_v3 VERSION AS OF 3;
```

Full rebuild rather than incremental merge is deliberate — the table is ~2.4k rows, and the lag and
rolling columns depend on neighbouring rows, so a partial update would produce wrong values at the
boundary.

In [0]:
DESCRIPTION = (
    'v3 — wide multi-station features predicting Bad (PRCP > 0.5mm) at Midway t+1. '
    'One row per DATE, not per station-day: `station` is a constant naming the prediction '
    'target, retained so the primary key matches v1/v2. Other stations appear as IATA-prefixed '
    'columns (RFD_, MSP_, MKE_, DBQ_, ORD_, IND_). See data_pipelines/11_features_nightly.ipynb.'
)

if spark.catalog.tableExists(FT3):
    # write_table, not DROP + create_table — this appends a Delta version instead of resetting
    # history to zero.
    fe.write_table(name=FT3, df=v3, mode='overwrite')
    print(f'overwrote {FT3} (new Delta version)')
else:
    fe.create_table(name=FT3, primary_keys=['date', 'station'], timestamp_keys=['date'],
                    df=v3, description=DESCRIPTION)
    print(f'created {FT3}')

display(spark.sql(f'DESCRIBE HISTORY {FT3}').select('version', 'timestamp', 'operation').limit(5))

## 4. Checks

The number that matters most is the last one. `03_baseline_model.ipynb` calls `.dropna()` on the
training set, so **every null column costs whole rows**. With one station's outage able to void a
date entirely, it is worth knowing the real training-set size before wondering why the metrics moved.

In [0]:
v3_tbl = spark.table(FT3)
total = v3_tbl.count()
print(f'{total} rows, {len(v3_tbl.columns)} columns')

# Primary key has to be unique or the feature lookup fans out.
dupes = v3_tbl.groupBy('date').count().filter('count > 1').count()
assert dupes == 0, f'{dupes} duplicate dates in {FT3}'

cols = [c for c in v3_tbl.columns if c not in ('date', 'station')]
null_rates = (v3_tbl
              .select([F.round(F.avg(F.col(c).isNull().cast('int')), 3).alias(c) for c in cols])
              .toPandas().T.rename(columns={0: 'null_rate'})
              .sort_values('null_rate', ascending=False))
print()
print(null_rates[null_rates['null_rate'] > 0].to_string() or '(no nulls)')

complete = v3_tbl.dropna().count()
print()
print(f'{complete} of {total} rows survive dropna() ({complete / total:.1%})')
print()
print('If that share is low, the fix is an imputer in 03\'s sklearn Pipeline rather than')
print('dropna() — a single upstream station missing one day should not void the whole date.')

---

**Wiring v3 into `03_baseline_model.ipynb`:** point `FT` at `weather_daily_v3`, set `FEATURES` to the
column list printed in section 2, and tag the run `feature_set_version: v3`. The existing
`FeatureLookup(lookup_key=['station'], timestamp_lookup_key='date')` works unchanged — that is the
reason for the constant `station` column.

Two things to expect when you do:

- **Swap `.dropna()` for an imputer.** With this many columns, one station's gap voids an entire
  date. `SimpleImputer(strategy='median')` in front of the scaler costs one line.
- **Feature selection is doing real work here.** ~40 features against ~475 minority events is at the
  edge of what logistic regression supports. An L1 penalty or an explicit selection step is the
  honest way to cut it down, and the comparison against v2 is more interesting with it than
  without.

**Do not benchmark against `MODEL_CARD.md`.** Those metrics are for the same-day task from
`train_baseline.py`. The comparable numbers are the v1 and v2 runs in the MLflow experiment.